In [ ]:
# Робота з табличними даними та математичними операціями
import numpy as np
import pandas as pd

# Візуалізація даних і результатів
import matplotlib.pyplot as plt
import seaborn as sns

# Підготовка даних та оцінювання моделі
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Створення і навчання нейронної мережі
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Допоміжні бібліотеки
from copy import deepcopy
import random

In [2]:
# Робимо результати експерименту відтворюваними
RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# Налаштовуємо зовнішній вигляд графіків
sns.set_theme(style="whitegrid")

print(f"PyTorch version: {torch.__version__}")
print("Libraries imported successfully.")

PyTorch version: 2.10.0+cpu
Libraries imported successfully.


In [ ]:
from pathlib import Path

# У Kaggle дані зберігаються в /kaggle/input, а локально — біля ноутбука.
KAGGLE_INPUT_DIR = Path("/kaggle/input")

if KAGGLE_INPUT_DIR.exists():
    csv_files = list(KAGGLE_INPUT_DIR.rglob("*.csv"))
else:
    csv_files = list(Path.cwd().glob("*.csv"))

print("Found CSV files:")
for file_path in csv_files:
    print(file_path)

In [ ]:
if len(csv_files) != 1:
    raise ValueError(
        f"Expected exactly one CSV file, but found {len(csv_files)}. "
        "Please select the concrete dataset file explicitly."
    )
DATA_PATH = csv_files[0]

df = pd.read_csv(DATA_PATH)

print(f"Dataset path: {DATA_PATH}")
print(f"Dataset shape: {df.shape}")

In [ ]:
print("Column names:")

for index, column in enumerate(df.columns):
    print(f"{index}: {column}")

In [ ]:
target_column = df.columns[-1]

print(f"Target column: {target_column}")

In [ ]:
# X містить усі вхідні ознаки, крім цільової колонки
X = df.drop(columns=[target_column]).copy()

# y містить значення міцності бетону, які модель повинна передбачити
y = df[target_column].copy()

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

display(X.head())
display(y.head())

In [ ]:
# Спочатку відкладаємо 20% усіх даних для фінального тестування.
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE
)

# Потім відокремлюємо 20% від навчальної частини для валідації.
# Підсумкове співвідношення: 64% train, 16% validation, 20% test.
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.2,
    random_state=RANDOM_STATE
)

print(f"Training features shape:   {X_train.shape}")
print(f"Validation features shape: {X_val.shape}")
print(f"Test features shape:       {X_test.shape}")
print(f"Training target shape:     {y_train.shape}")
print(f"Validation target shape:   {y_val.shape}")
print(f"Test target shape:         {y_test.shape}")

In [ ]:
# StandardScaler вивчає параметри ТІЛЬКИ на навчальній вибірці.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Валідаційні й тестові дані перетворюємо без повторного fit.
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"Scaled training data shape:   {X_train_scaled.shape}")
print(f"Scaled validation data shape: {X_val_scaled.shape}")
print(f"Scaled test data shape:       {X_test_scaled.shape}")

In [ ]:
print("Training feature means after scaling:")
print(np.round(X_train_scaled.mean(axis=0), 4))

print("\nTraining feature standard deviations after scaling:")
print(np.round(X_train_scaled.std(axis=0), 4))

# Validation і test не зобов'язані мати точні 0 та 1, бо scaler не навчався на них.
print("\nValidation feature means after scaling:")
print(np.round(X_val_scaled.mean(axis=0), 4))

In [ ]:
class ConcreteStrengthModel(nn.Module):
    def __init__(self, input_size):
        super().__init__()

        self.hidden_layer_1 = nn.Linear(input_size, 64)
        self.hidden_layer_2 = nn.Linear(64, 32)
        self.output_layer = nn.Linear(32, 1)

        self.activation = nn.ReLU()

    def forward(self, x):
        x = self.hidden_layer_1(x)
        x = self.activation(x)

        x = self.hidden_layer_2(x)
        x = self.activation(x)

        x = self.output_layer(x)

        return x

In [ ]:
# Кількість колонок у нормалізованій навчальній матриці ознак.
input_size = X_train_scaled.shape[1]

# Створюємо модель до початку навчання.
model = ConcreteStrengthModel(input_size=input_size)

print(f"Number of input features: {input_size}")
print(model)

In [ ]:
total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print(f"Number of trainable parameters: {total_parameters}")

In [ ]:
LEARNING_RATE = 0.01
BATCH_SIZE = 32
NUM_EPOCHS = 300

# Якщо validation loss не покращується 40 епох, навчання зупиняється.
EARLY_STOPPING_PATIENCE = 40
MIN_DELTA = 1e-4

In [ ]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

y_train_tensor = torch.tensor(
    y_train.to_numpy(), dtype=torch.float32
).reshape(-1, 1)

y_val_tensor = torch.tensor(
    y_val.to_numpy(), dtype=torch.float32
).reshape(-1, 1)

y_test_tensor = torch.tensor(
    y_test.to_numpy(), dtype=torch.float32
).reshape(-1, 1)

In [ ]:
print(f"X_train tensor: shape={X_train_tensor.shape}, dtype={X_train_tensor.dtype}")
print(f"y_train tensor: shape={y_train_tensor.shape}, dtype={y_train_tensor.dtype}")
print(f"X_val tensor:   shape={X_val_tensor.shape}, dtype={X_val_tensor.dtype}")
print(f"y_val tensor:   shape={y_val_tensor.shape}, dtype={y_val_tensor.dtype}")
print(f"X_test tensor:  shape={X_test_tensor.shape}, dtype={X_test_tensor.dtype}")
print(f"y_test tensor:  shape={y_test_tensor.shape}, dtype={y_test_tensor.dtype}")

In [ ]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [ ]:
print(f"Training examples:   {len(train_dataset)}")
print(f"Validation examples: {len(val_dataset)}")
print(f"Test examples:       {len(test_dataset)}")

In [ ]:
features, target = train_dataset[0]

print(f"Feature shape: {features.shape}")
print(f"Target shape:  {target.shape}")

In [ ]:
train_generator = torch.Generator().manual_seed(RANDOM_STATE)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=train_generator
)

# Порядок validation/test не змінюємо, бо на них ваги не оновлюються.
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [ ]:
batch_features, batch_targets = next(iter(train_loader))

print(f"Feature batch shape: {batch_features.shape}")
print(f"Target batch shape:  {batch_targets.shape}")

In [ ]:
criterion = nn.MSELoss()

In [ ]:
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=LEARNING_RATE
)

In [ ]:
print(f"Loss function: {criterion}")
print(f"Optimizer: {optimizer.__class__.__name__}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Maximum number of epochs: {NUM_EPOCHS}")
print(f"Early stopping patience: {EARLY_STOPPING_PATIENCE}")

In [ ]:
train_loss_history = []
val_loss_history = []

best_val_loss = float("inf")
best_epoch = 0
best_model_state = None
epochs_without_improvement = 0

for epoch in range(NUM_EPOCHS):
    # Навчання: градієнти обчислюються, а ваги моделі оновлюються.
    model.train()
    train_running_loss = 0.0

    for batch_features, batch_targets in train_loader:
        optimizer.zero_grad()
        predictions = model(batch_features)
        loss = criterion(predictions, batch_targets)
        loss.backward()
        optimizer.step()

        train_running_loss += loss.item() * batch_features.size(0)

    train_epoch_loss = train_running_loss / len(train_loader.dataset)
    train_loss_history.append(train_epoch_loss)

    # Валідація: ваги не змінюються, градієнти не потрібні.
    model.eval()
    val_running_loss = 0.0

    with torch.no_grad():
        for batch_features, batch_targets in val_loader:
            val_predictions = model(batch_features)
            val_loss = criterion(val_predictions, batch_targets)
            val_running_loss += val_loss.item() * batch_features.size(0)

    val_epoch_loss = val_running_loss / len(val_loader.dataset)
    val_loss_history.append(val_epoch_loss)

    # Зберігаємо ваги лише тоді, коли validation MSE стала меншою.
    if val_epoch_loss < best_val_loss - MIN_DELTA:
        best_val_loss = val_epoch_loss
        best_epoch = epoch + 1
        best_model_state = deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if (epoch + 1) % 25 == 0 or epoch == 0:
        print(
            f"Epoch [{epoch + 1:3d}/{NUM_EPOCHS}], "
            f"Training MSE: {train_epoch_loss:.4f}, "
            f"Validation MSE: {val_epoch_loss:.4f}"
        )

    # Зупиняємося, якщо якість на validation довго не покращується.
    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"Early stopping at epoch {epoch + 1}.")
        break

if best_model_state is None:
    raise RuntimeError("Training failed: no finite validation loss was obtained.")

# Для фінального оцінювання відновлюємо найкращі, а не останні ваги.
model.load_state_dict(best_model_state)

print(f"Best epoch: {best_epoch}")
print(f"Best validation MSE: {best_val_loss:.4f} MPa²")

In [ ]:
epochs = np.arange(1, len(train_loss_history) + 1)

plt.figure(figsize=(10, 5))
plt.plot(epochs, train_loss_history, color="royalblue", label="Training MSE")
plt.plot(epochs, val_loss_history, color="darkorange", label="Validation MSE")
plt.axvline(
    best_epoch,
    color="red",
    linestyle="--",
    label=f"Best epoch: {best_epoch}"
)

plt.xlabel("Epoch")
plt.ylabel("Mean Squared Error, MPa²")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
model.eval()

with torch.no_grad():
    val_predictions_tensor = model(X_val_tensor)
    test_predictions_tensor = model(X_test_tensor)

y_val_values = y_val_tensor.cpu().numpy().reshape(-1)
val_predictions = val_predictions_tensor.cpu().numpy().reshape(-1)

y_test_values = y_test_tensor.cpu().numpy().reshape(-1)
test_predictions = test_predictions_tensor.cpu().numpy().reshape(-1)

print(f"Validation predictions: {len(val_predictions)}")
print(f"Test predictions:       {len(test_predictions)}")

In [ ]:
val_mse = mean_squared_error(y_val_values, val_predictions)
val_rmse = np.sqrt(val_mse)
val_mae = mean_absolute_error(y_val_values, val_predictions)
val_r2 = r2_score(y_val_values, val_predictions)

test_mse = mean_squared_error(y_test_values, test_predictions)
test_rmse = np.sqrt(test_mse)
test_mae = mean_absolute_error(y_test_values, test_predictions)
test_r2 = r2_score(y_test_values, test_predictions)

metrics_df = pd.DataFrame(
    {
        "MSE, MPa²": [val_mse, test_mse],
        "RMSE, MPa": [val_rmse, test_rmse],
        "MAE, MPa": [val_mae, test_mae],
        "R²": [val_r2, test_r2]
    },
    index=["Validation", "Test"]
)

display(metrics_df.round(4))

In [ ]:
predictions_df = pd.DataFrame({
    "Actual strength": y_test_values,
    "Predicted strength": test_predictions
})

predictions_df["Absolute error"] = np.abs(
    predictions_df["Actual strength"]
    - predictions_df["Predicted strength"]
)

predictions_df.head(10)

In [ ]:
minimum_value = min(
    y_test_values.min(),
    test_predictions.min()
)

maximum_value = max(
    y_test_values.max(),
    test_predictions.max()
)

plt.figure(figsize=(8, 6))

plt.scatter(
    y_test_values,
    test_predictions,
    alpha=0.7,
    color="royalblue"
)

plt.plot(
    [minimum_value, maximum_value],
    [minimum_value, maximum_value],
    color="red",
    linestyle="--",
    label="Perfect prediction"
)

plt.xlabel("Actual Strength, MPa")
plt.ylabel("Predicted Strength, MPa")
plt.title("Actual vs Predicted Concrete Strength")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Межі для графіка фактичних і прогнозованих тестових значень.
plot_min = min(y_test_values.min(), test_predictions.min())
plot_max = max(y_test_values.max(), test_predictions.max())

# Сортування використовується лише для наочного порівняння кривих.
sort_indices = np.argsort(y_test_values)
actual_sorted = y_test_values[sort_indices]
predicted_sorted = test_predictions[sort_indices]

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(21, 6))

# 1. Контроль перенавчання за train/validation loss.
axes[0].plot(epochs, train_loss_history, color="royalblue", label="Training MSE")
axes[0].plot(epochs, val_loss_history, color="darkorange", label="Validation MSE")
axes[0].axvline(best_epoch, color="red", linestyle="--", label="Best epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE, MPa²")
axes[0].set_title("Training and Validation Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Фактичні та прогнозовані тестові значення.
axes[1].scatter(y_test_values, test_predictions, color="darkorange", alpha=0.7)
axes[1].plot(
    [plot_min, plot_max],
    [plot_min, plot_max],
    color="red",
    linestyle="--",
    linewidth=2,
    label="Perfect prediction"
)
axes[1].set_xlabel("Actual Strength, MPa")
axes[1].set_ylabel("Predicted Strength, MPa")
axes[1].set_title("Test: Actual vs Predicted")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 3. Відсортоване порівняння тестових значень.
axes[2].plot(actual_sorted, color="royalblue", linewidth=2, label="Actual")
axes[2].plot(
    predicted_sorted,
    color="darkorange",
    linewidth=2,
    alpha=0.8,
    label="Predicted"
)
axes[2].set_xlabel("Test Samples Sorted by Actual Strength")
axes[2].set_ylabel("Concrete Strength, MPa")
axes[2].set_title("Test: Sorted Actual and Predicted")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

fig.suptitle("Concrete Strength Model Results", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
initial_train_loss = train_loss_history[0]
best_train_loss = train_loss_history[best_epoch - 1]
final_train_loss = train_loss_history[-1]
final_val_loss = val_loss_history[-1]

loss_reduction = (
    (initial_train_loss - best_train_loss)
    / initial_train_loss
    * 100
)

generalization_gap = best_val_loss - best_train_loss

print(f"Initial training MSE:       {initial_train_loss:.4f} MPa²")
print(f"Training MSE at best epoch: {best_train_loss:.4f} MPa²")
print(f"Best validation MSE:        {best_val_loss:.4f} MPa²")
print(f"Final training MSE:         {final_train_loss:.4f} MPa²")
print(f"Final validation MSE:       {final_val_loss:.4f} MPa²")
print(f"Best epoch:                 {best_epoch}")
print(f"Training loss reduction:    {loss_reduction:.2f}%")
print(f"Generalization gap:         {generalization_gap:.4f} MPa²")

In [ ]:
print("ВИСНОВКИ")
print("-" * 70)
print(
    "Дані було розділено на train/validation/test до стандартизації "
    "та до початку навчання."
)
print(
    "StandardScaler навчався лише на train, тому витоку інформації "
    "з validation або test не відбулося."
)
print(
    f"Найкращу версію моделі вибрано за мінімальною validation MSE "
    f"на епосі {best_epoch}."
)
print(
    f"Validation: RMSE={val_rmse:.4f} MPa, MAE={val_mae:.4f} MPa, "
    f"R²={val_r2:.4f}."
)
print(
    f"Test: RMSE={test_rmse:.4f} MPa, MAE={test_mae:.4f} MPa, "
    f"R²={test_r2:.4f}."
)
print(
    "Early stopping і відновлення найкращих ваг використано для "
    "контролю перенавчання."
)